In [11]:
%pip install -q -U pymupdf beautifulsoup4

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [12]:
from pathlib import Path
import json
import shutil

import fitz
from bs4 import BeautifulSoup, NavigableString

In [296]:
ROOT = Path.home() / "gemini HTML output"

LANGUAGE = "Assamese"
DOC_NAME = "Lentil_KVK , Thoubal ,Manipur"

WORKDIR_ROOT = ROOT / "Workdir" / LANGUAGE / DOC_NAME

START_PAGE = 1
END_PAGE = 2

OVERWRITE_EXISTING = True

print("Workdir root:", WORKDIR_ROOT)
print("Exists:", WORKDIR_ROOT.exists())

Workdir root: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur
Exists: True


In [297]:
def extract_embedded_images(pdf_path: Path, out_dir: Path):
    doc = fitz.open(pdf_path)
    page = doc[0]

    image_list = page.get_images(full=True)
    extracted = []
    seen_xrefs = set()

    out_dir.mkdir(parents=True, exist_ok=True)

    for idx, img in enumerate(image_list, start=1):
        xref = img[0]

        if xref in seen_xrefs:
            continue
        seen_xrefs.add(xref)

        img_info = doc.extract_image(xref)
        img_bytes = img_info["image"]
        ext = img_info.get("ext", "png")

        out_path = out_dir / f"image_{len(extracted)+1}.{ext}"
        out_path.write_bytes(img_bytes)

        extracted.append({
            "index": len(extracted) + 1,
            "xref": xref,
            "path": out_path,
        })

    doc.close()
    return extracted

In [298]:
def inject_images_into_html(html_path: Path, embedded_images, final_html_path: Path):
    html_text = html_path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html_text, "html.parser")

    placeholders = soup.find_all("figure", class_="image-placeholder")
    placeholder_count = len(placeholders)
    image_count = len(embedded_images)

    for idx, fig in enumerate(placeholders, start=1):
        if idx > image_count:
            continue

        img_meta = embedded_images[idx - 1]
        img_rel_path = f"images/{img_meta['path'].name}"

        for content in list(fig.contents):
            if isinstance(content, NavigableString) and "[IMAGE]" in str(content):
                content.extract()

        img_tag = soup.new_tag("img")
        img_tag["src"] = img_rel_path
        img_tag["alt"] = f"Image {idx}"
        img_tag["style"] = "max-width: 100%; height: auto; display: block; margin: 0 auto;"
        fig.insert(0, img_tag)

        existing_classes = fig.get("class", [])
        fig["class"] = [c for c in existing_classes if c != "image-placeholder"]

    final_html_path.write_text(str(soup), encoding="utf-8")
    print(f"Saved final HTML with images: {final_html_path}")

    return {
        "placeholder_count": placeholder_count,
        "embedded_image_count": image_count,
        "matched_count": min(placeholder_count, image_count),
    }

In [299]:
summary = []

for page_num in range(START_PAGE, END_PAGE + 1):
    page_name = f"page_{page_num:03d}"
    page_dir = WORKDIR_ROOT / page_name
    pdf_path = page_dir / f"{page_name}.pdf"
    html_path = page_dir / "translated.html"
    images_dir = page_dir / "images"
    final_html_path = page_dir / "final_with_images.html"
    err_path = page_dir / "image_injection_error.txt"

    if not pdf_path.exists():
        msg = f"PDF not found: {pdf_path}"
        print(msg)
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": msg,
        })
        continue

    if not html_path.exists():
        msg = f"translated.html not found: {html_path}"
        print(msg)
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": msg,
        })
        continue

    if final_html_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping {page_name}: final_with_images.html already exists")
        summary.append({
            "page": page_name,
            "status": "skipped",
            "reason": "final_with_images.html already exists",
        })
        continue

    print(f"\n=== Processing {page_name} ===")

    try:
        if images_dir.exists() and OVERWRITE_EXISTING:
            shutil.rmtree(images_dir)

        embedded_images = extract_embedded_images(pdf_path, images_dir)
        injection_result = inject_images_into_html(html_path, embedded_images, final_html_path)

        summary.append({
            "page": page_name,
            "status": "success",
            "embedded_image_count": len(embedded_images),
            "placeholder_count": injection_result["placeholder_count"],
            "matched_count": injection_result["matched_count"],
            "output_file": str(final_html_path),
        })
        print(
            f"Saved: {final_html_path} | "
            f"images={len(embedded_images)} placeholders={injection_result['placeholder_count']} "
            f"matched={injection_result['matched_count']}"
        )

    except Exception as e:
        err = str(e)
        err_path.write_text(err, encoding="utf-8")
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": err,
            "error_file": str(err_path),
        })
        print(f"Failed: {page_name} -> {err}")


=== Processing page_001 ===
Saved final HTML with images: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur/page_001/final_with_images.html
Saved: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur/page_001/final_with_images.html | images=1 placeholders=5 matched=1

=== Processing page_002 ===
Saved final HTML with images: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur/page_002/final_with_images.html
Saved: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur/page_002/final_with_images.html | images=1 placeholders=1 matched=1
